In [2]:
!pip install -q -U google-genai


In [4]:
from google import genai
from google.colab import userdata
from google.genai import types
from pydantic import BaseModel
from typing import Optional
import json

client=genai.Client(api_key=userdata.get("GEMINI_API_KEYS"))
MODEL="gemini-3.5-flash-lite"

In [9]:
user_input=input("Enter your info name,experience,skills,job role")
prompt=f"""
Extract name,experince,skills,job role from user input={user_input}

based on job role generate skills required and match skills of user and job role generate skills and provide missing_skills,score.
return JSON format containing keys (name,experince,job_role,skils(list),missing_skills,score(out of 100),valid(if score>60 valid=true else valid=false)).
"""
class job_matcher(BaseModel):
  name:Optional[str]
  experience:Optional[int]
  job_role:Optional[str]
  skills:Optional[list]
  missing_skills:Optional[list]
  score:Optional[int]
  valid:Optional[bool]
response=client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.0,
        max_output_tokens=2048,
        system_instruction="Your job matching checker,Reply with (Hi,I am your job description matcher)",
        response_mime_type="application/json",
        response_schema=job_matcher,
        thinking_config=types.ThinkingConfig(thinking_level='low')
    )
)

data=json.loads(response.text)
keys=['name','experience','job_role','skills','missing_skills','score','valid']
for key in keys:
  data.setdefault(key,None)

print(json.dumps(data,indent=4))


Enter your info name,experience,skills,job rolei am kushal having 3years of experince at infosis i know python dsa os oops with c++ dbms applyign for role of SDE
{
    "name": "kushal",
    "experience": 3,
    "job_role": "SDE",
    "skills": [
        "python",
        "dsa",
        "os",
        "oops",
        "c++",
        "dbms"
    ],
    "missing_skills": [
        "system design",
        "cloud computing",
        "git"
    ],
    "score": 70,
    "valid": true
}
